## 1. Environment Setup

Run the cell below. On first run this opens a browser window/tab for authentication — sign in with the Google account that was added to the shared training project.

**If you get a permission error on `Initialize`**: Flag it immediately rather than debugging — it's a project-access issue, not a code issue.

In [ ]:
import ee
import geemap

# Swap this for the shared training project ID given in the setup slides
PROJECT_ID = "riftwaters"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("✅ Earth Engine initialized")

## 2. Geometries

`ee.Geometry` defines *where* — every acquisition and filter operation needs one. Common constructors: `ee.Geometry.Point`, `ee.Geometry.Polygon`, `ee.Geometry.Rectangle`.

Below is the **shared training ROI** (Lake Naivasha) and date range. Copy this cell as-is into every notebook this week.

In [ ]:
# --- Shared training config: reuse this cell in every notebook ---
ROI = ee.Geometry.Polygon([
            [36.244378206729166,-0.838827881647561],
            [36.442132112979166,-0.838827881647561],
            [36.442132112979166,-0.6664947946163098],
            [36.244378206729166,-0.6664947946163098],
            [36.244378206729166,-0.838827881647561]     
        ])

START_DATE = "2026-01-01"
END_DATE = "2026-02-28"
REGION_NAME = "naivasha"

area = ROI.area().divide(1e6).getInfo()
print("ROI area (km²):", area)

## 3. ImageCollection vs Image, and filtering

- `ee.Image`: a single satellite image.
- `ee.ImageCollection`: a stack of images.



In [ ]:
# An example of an Image collection
s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ROI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 30))
)

count = s2_collection.size().getInfo()

s2_images = s2_collection.aggregate_array('system:index').getInfo() #This will return a list of all the scene indices in the collection
print(f"Found {count} Sentinel-2 scenes over Naivasha, {START_DATE} → {END_DATE}")
print(f"Scenes: {s2_images}")

In [ ]:
# Inspect one image by printing the available bands
first_image = s2_collection.first()
print("Bands:", first_image.bandNames().getInfo())
print("Date:", ee.Date(first_image.get('system:time_start')).format('YYYY-MM-dd').getInfo())

## 4. Visualizing with geemap

`geemap` gives us an interactive map inside the notebook so we can sanity-check what we've filtered before doing any export or processing. This is the fastest way to catch a wrong ROI or empty collection.

In [ ]:
composite = s2_collection.median().clip(ROI)

vis_params = {
    "bands": ["B4", "B3", "B2"],  # true color
    "min": 0,
    "max": 3000,
}

Map = geemap.Map()
Map.centerObject(ROI, 12)
Map.addLayer(composite, vis_params, "S2 median composite")
Map.addLayer(ROI, {"color": "red"}, "Training ROI", opacity=0.3)
Map

Now lets select 1 scene from the scenes in the image collection


In [ ]:
#Try images from 0-17 to see the difference (0-17 are the indices of the images in the collection)
scene = s2_collection.filter(ee.Filter.eq('system:index', s2_images[7]))
Map = geemap.Map()
Map.centerObject(ROI, 10)
Map.addLayer(scene, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 1000}, "scene")
Map

In [ ]:
#An example of an ee.Image image
s2_image = ee.Image("S2B_MSIL1C_20260226T073759_N0512_R092_T36NZF_20260226T111448")
Map = geemap.Map()
Map.centerObject(ROI, 10)
Map.addLayer(s2_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 1000}, "s2_image")
Map

## Exercise 1.1

Using the pattern above:

1. Pick a **different lake** from the list below and build its ROI (coordinates given).
2. Filter the Sentinel-2 collection over that ROI for the **same date range** (`START_DATE`/`END_DATE`).
3. Print how many scenes were found.
4. Display a true-color composite on a `geemap.Map()`.

```python
# Nakuru ROI, for reference
NAKURU_ROI = ee.Geometry.Polygon([
    [36.045150257654164, -0.41882934935973415],
    [36.13441417366979, -0.41882934935973415],
    [36.13441417366979, -0.29935539913088666],
    [36.045150257654164, -0.29935539913088666],
    [36.045150257654164, -0.41882934935973415],
])

ELEMENTAITA_ROI = ee.Geometry.Polygon([
            [36.20503005704847,-0.4844042871968538],
            [36.27541122159925,-0.4844042871968538],
            [36.27541122159925,-0.3992627836576272],
            [36.20503005704847,-0.3992627836576272],
            [36.20503005704847,-0.4844042871968538]
        ])
BARINGO_ROI = ee.Geometry.Polygon([
            [35.98987673288601,0.4361077040885743],
            [36.180764184057885,0.4361077040885743],
            [36.180764184057885,0.7574399496732773],
            [35.98987673288601,0.7574399496732773],
            [35.98987673288601,0.4361077040885743]
        ])

```

Try to do it without re-reading the cells above — you'll need this exact filter → count → visualize pattern constantly for the rest of the week.

In [ ]:
# Your solution here
